# Model 5 — MobileNetV2 Transfer Learning

**Group**: Group 14  
**Members**: *(update with real names)*  
**Model owner**: P5  
**Architecture**: MobileNetV2 (ImageNet weights, include_top=False) + GlobalAveragePooling2D + Dense(128) + Dropout(0.3) + sigmoid  
**Dataset**: NIH Malaria Cell Images — Parasitized vs Uninfected  
**Date**: June 2026

---
This notebook applies MobileNetV2 to malaria cell binary classification via transfer learning.  
MobileNetV2 uses depthwise separable convolutions and inverted residuals, making it significantly  
lighter than VGG16 or ResNet50 while remaining competitive in accuracy — ideal for deployment  
in resource-constrained clinical settings. The `alpha` width multiplier is explored as an  
architectural variable, and Experiment 7 compares against EfficientNetB0 as an alternative base.

## 1. Environment Setup
Install dependencies, set all random seeds for reproducibility, and verify GPU availability.

In [ ]:
!pip install -q kagglehub tqdm

import os, sys, random
import numpy as np
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU available:', gpus if gpus else 'None — training on CPU')

## 2. Dataset Download
Download the NIH Malaria dataset via `kagglehub`. The inner `cell_images` folder is targeted explicitly to avoid the nested directory conflict on platforms such as Deepnote.

In [ ]:
import kagglehub

path = kagglehub.dataset_download('iarunava/cell-images-for-detecting-malaria')
print('Downloaded to:', path)

# Skip outer cell_images/ folder — only match directory containing BOTH class folders and nothing else
DATA_DIR = None
for root, dirs, _ in os.walk(path):
    if 'Parasitized' in dirs and 'Uninfected' in dirs and 'cell_images' not in dirs:
        DATA_DIR = root
        break

assert DATA_DIR is not None, 'Could not locate Parasitized/Uninfected folders'
print('DATA_DIR:', DATA_DIR)
print('Parasitized images:', len(os.listdir(os.path.join(DATA_DIR, 'Parasitized'))))
print('Uninfected images: ', len(os.listdir(os.path.join(DATA_DIR, 'Uninfected'))))

## 3. Shared Helper Functions
All helper functions are defined inline — this notebook runs independently on any platform without needing `utils.py`.  
**Note**: MobileNetV2 requires raw pixel values [0, 255] — `mobilenet_v2.preprocess_input` is applied inside the model, so `load_dataset` is called with `normalise=False`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve,
)

# ── Constants
IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
SEED        = 42
CLASS_NAMES = ['Parasitized', 'Uninfected']
TRAIN_SPLIT = 0.80
VAL_SPLIT   = 0.10
TEST_SPLIT  = 0.10

os.makedirs('figures',     exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# ── Augmentation layer
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name='data_augmentation')

# ── Dataset loader (normalise=False for pretrained models)
def load_dataset(data_dir, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, normalise=True):
    full_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        labels='inferred',
        label_mode='binary',
        class_names=CLASS_NAMES,
        image_size=image_size,
        batch_size=None,
        shuffle=True,
        seed=SEED,
    )
    total   = sum(1 for _ in full_ds)
    n_train = int(total * TRAIN_SPLIT)
    n_val   = int(total * VAL_SPLIT)
    train_ds  = full_ds.take(n_train)
    remaining = full_ds.skip(n_train)
    val_ds    = remaining.take(n_val)
    test_ds   = remaining.skip(n_val)
    AUTOTUNE  = tf.data.AUTOTUNE
    cast_fn = (lambda img, lbl: (tf.cast(img, tf.float32) / 255.0, lbl)) if normalise \
              else (lambda img, lbl: (tf.cast(img, tf.float32), lbl))
    train_ds = (train_ds.map(cast_fn, num_parallel_calls=AUTOTUNE)
                .cache().shuffle(1000, seed=SEED).batch(batch_size).prefetch(AUTOTUNE))
    val_ds   = (val_ds.map(cast_fn, num_parallel_calls=AUTOTUNE)
                .cache().batch(batch_size).prefetch(AUTOTUNE))
    test_ds  = (test_ds.map(cast_fn, num_parallel_calls=AUTOTUNE)
                .cache().batch(batch_size).prefetch(AUTOTUNE))
    return train_ds, val_ds, test_ds

# ── Metrics
def evaluate_model(model, test_ds):
    y_true, y_pred_prob = [], []
    for images, labels in test_ds:
        preds = model(images, training=False).numpy().flatten()
        y_pred_prob.extend(preds)
        y_true.extend(labels.numpy().flatten())
    y_true      = np.array(y_true)
    y_pred_prob = np.array(y_pred_prob)
    y_pred      = (y_pred_prob >= 0.5).astype(int)
    return {
        'accuracy':  round(accuracy_score(y_true, y_pred),     4),
        'precision': round(precision_score(y_true, y_pred),    4),
        'recall':    round(recall_score(y_true, y_pred),       4),
        'f1':        round(f1_score(y_true, y_pred),           4),
        'auc':       round(roc_auc_score(y_true, y_pred_prob), 4),
        'y_true':    y_true,
        'y_pred':    y_pred,
        'y_prob':    y_pred_prob,
    }

# ── Learning curves
def plot_learning_curves(history, experiment_name, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Learning curves — {experiment_name}', fontsize=14, fontweight='bold')
    axes[0].plot(history.history['loss'],     label='Train loss',     linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Val loss',       linewidth=2, linestyle='--')
    axes[0].set_title('Loss over epochs'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history.history['accuracy'],     label='Train accuracy', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='Val accuracy',   linewidth=2, linestyle='--')
    axes[1].set_title('Accuracy over epochs'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    final_train = history.history['accuracy'][-1]
    final_val   = history.history['val_accuracy'][-1]
    gap = final_train - final_val
    if gap > 0.05:
        print(f'Overfitting detected: train {final_train:.3f} vs val {final_val:.3f} (gap={gap:.3f})')
    elif gap < -0.02:
        print(f'Underfitting detected: train {final_train:.3f} vs val {final_val:.3f}')
    else:
        print(f'Good fit: train {final_train:.3f} vs val {final_val:.3f} (gap={gap:.3f})')

# ── Confusion matrix
def plot_confusion_matrix(metrics_dict, class_names, experiment_name, save_path=None):
    cm  = confusion_matrix(metrics_dict['y_true'], metrics_dict['y_pred'])
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax, linewidths=0.5)
    ax.set_title(f'Confusion matrix — {experiment_name}', fontsize=13, fontweight='bold', pad=14)
    ax.set_ylabel('True label', fontsize=11); ax.set_xlabel('Predicted label', fontsize=11)
    tn, fp, fn, tp = cm.ravel()
    ax.text(0.5, -0.12,
            f'TP={tp}  TN={tn}  FP={fp}  FN={fn}  |  Sensitivity={tp/(tp+fn):.3f}  Specificity={tn/(tn+fp):.3f}',
            ha='center', va='top', transform=ax.transAxes, fontsize=10, color='dimgray')
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# ── ROC curve
def plot_roc_curve(metrics_dict, experiment_name, save_path=None):
    fpr, tpr, _ = roc_curve(metrics_dict['y_true'], metrics_dict['y_prob'])
    auc_val = metrics_dict['auc']
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, color='royalblue', lw=2, label=f'ROC curve (AUC = {auc_val:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
    ax.fill_between(fpr, tpr, alpha=0.08, color='royalblue')
    ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
    ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=11)
    ax.set_title(f'ROC Curve — {experiment_name}', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# ── Results table
def build_results_table(experiments_list):
    df = pd.DataFrame(experiments_list)
    df = df[['exp_num','description','accuracy','precision','recall','f1','auc','epochs','notes']]
    df.columns = ['Exp #','Description','Accuracy','Precision','Recall','F1','AUC','Epochs','Notes']
    return df.sort_values('F1', ascending=False).reset_index(drop=True)

# ── Callbacks
def get_callbacks(model_name, experiment_num, patience_es=8, patience_lr=4):
    os.makedirs('checkpoints', exist_ok=True)
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=patience_es, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=patience_lr, min_lr=1e-8, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f'checkpoints/{model_name}_exp{experiment_num}.h5',
            monitor='val_accuracy', save_best_only=True, verbose=0),
    ]

# ── Error analysis
def error_analysis(model, test_ds, class_names, n_samples=12):
    misclassified_images, misclassified_labels, misclassified_preds = [], [], []
    for images, labels in test_ds:
        preds        = model(images, training=False).numpy().flatten()
        pred_classes = (preds >= 0.5).astype(int)
        labels_np    = labels.numpy().astype(int).flatten()
        images_np    = images.numpy()
        mask         = pred_classes != labels_np
        misclassified_images.extend(images_np[mask])
        misclassified_labels.extend(labels_np[mask])
        misclassified_preds.extend(preds[mask])
        if len(misclassified_images) >= n_samples:
            break
    n   = min(n_samples, len(misclassified_images))
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    fig.suptitle('Error Analysis — Misclassified Samples', fontsize=14, fontweight='bold')
    for i, ax in enumerate(axes.flatten()[:n]):
        img = misclassified_images[i]
        if img.max() > 1.0:
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        ax.imshow(img)
        true_lbl = class_names[int(misclassified_labels[i])]
        pred_lbl = class_names[int(misclassified_preds[i] >= 0.5)]
        conf     = misclassified_preds[i] if pred_lbl == class_names[1] else 1 - misclassified_preds[i]
        ax.set_title(f'True: {true_lbl}\nPred: {pred_lbl} ({conf:.2f})', fontsize=9, color='red')
        ax.axis('off')
    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/P5_error_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Helper functions loaded.')

## 4. Load Dataset
MobileNetV2 requires 224×224 input. `normalise=False` preserves raw [0, 255] pixel values — `mobilenet_v2.preprocess_input` inside the model scales values to [−1, 1] as required by MobileNetV2's ImageNet-pretrained weights.

In [ ]:
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

train_ds, val_ds, test_ds = load_dataset(
    data_dir=DATA_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    normalise=False,  # mobilenet_v2.preprocess_input applied inside model
)

n_train = sum(1 for _ in train_ds) * BATCH_SIZE
n_val   = sum(1 for _ in val_ds)   * BATCH_SIZE
n_test  = sum(1 for _ in test_ds)  * BATCH_SIZE
print(f'Train: ~{n_train} | Val: ~{n_val} | Test: ~{n_test}')

## 5. Model Architecture
MobileNetV2 base (ImageNet weights, top excluded) with a lightweight head: GlobalAveragePooling2D → Dense(128) → Dropout(0.3) → sigmoid.  
The `alpha` parameter controls the width multiplier — smaller values reduce the number of filters at each layer, yielding a more compact model suitable for resource-constrained deployment.  
`mobilenet_v2.preprocess_input` scales [0, 255] → [−1, 1] as required by MobileNetV2.

In [ ]:
def build_mobilenetv2_model(freeze_base=True, alpha=1.0,
                            use_augmentation=False, fine_tune_last_n=None):
    """
    freeze_base      : freeze entire MobileNetV2 base
    alpha            : width multiplier (1.0 / 0.75 / 0.5) — controls model size
    use_augmentation : prepend data_augmentation layer
    fine_tune_last_n : int — unfreeze the last N layers of the base for fine-tuning
    """
    base_model = tf.keras.applications.MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3),
        alpha=alpha,
    )
    base_model.trainable = not freeze_base

    if fine_tune_last_n is not None:
        base_model.trainable = True
        for layer in base_model.layers[:-fine_tune_last_n]:
            layer.trainable = False
        trainable = sum(1 for l in base_model.layers if l.trainable)
        print(f'Fine-tuning last {fine_tune_last_n} layers ({trainable} trainable)')

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x      = inputs

    if use_augmentation:
        x = data_augmentation(x)

    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    return tf.keras.Model(inputs, outputs,
                          name=f'mobilenetv2_alpha{alpha}'), base_model

# Preview default architecture
m, _ = build_mobilenetv2_model(freeze_base=True, alpha=1.0)
m.summary()
del m

## 6. Experiment Tracking
All experiment results are appended to `results_log`. The final summary table is displayed after all 7 experiments.

In [ ]:
results_log = []  # Append one dict per experiment — never overwrite

---
## Experiment 1: Fully Frozen Base — Head Only (alpha=1.0)

**Hypothesis**: Training only the lightweight Dense(128) head while keeping all MobileNetV2 layers frozen establishes the transfer learning baseline. Despite its compact size, MobileNetV2's efficient depthwise separable convolutions should capture sufficient cell morphology features.

**Change made**: `freeze_base=True`, `alpha=1.0`, LR=1e-3, 10 epochs

In [ ]:
EXP_NUM         = 1
EXP_DESCRIPTION = 'Fully frozen base, alpha=1.0 — head only'
EPOCHS          = 10

model1, base1 = build_mobilenetv2_model(freeze_base=True, alpha=1.0)
model1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history1 = model1.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=get_callbacks('mobilenetv2', EXP_NUM), verbose=1,
)

metrics1 = evaluate_model(model1, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics1["accuracy"]}')
print(f'Precision: {metrics1["precision"]}')
print(f'Recall:    {metrics1["recall"]}')
print(f'F1-Score:  {metrics1["f1"]}')
print(f'AUC:       {metrics1["auc"]}')

plot_learning_curves(history1, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P5_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics1['accuracy'], 'precision': metrics1['precision'],
    'recall': metrics1['recall'], 'f1': metrics1['f1'], 'auc': metrics1['auc'],
    'epochs': len(history1.history['loss']), 'notes': '',
})

**Interpretation**: *(How did frozen MobileNetV2 features transfer to malaria cells? Compare convergence speed and F1 against frozen VGG16 and ResNet50.)*

---
## Experiment 2: Fine-tune Top 20 Layers

**Hypothesis**: Unfreezing the last 20 MobileNetV2 layers at a reduced LR (1e-5) allows the top inverted residual blocks to adapt to malaria cell features. MobileNetV2's lightweight structure means fine-tuning 20 layers is proportionally deeper than in VGG16 or ResNet50.

**Change made**: Two-stage — Stage 1: frozen head (10 epochs, LR=1e-3) → Stage 2: last 20 layers unfrozen (10 epochs, LR=1e-5)

In [ ]:
EXP_NUM         = 2
EXP_DESCRIPTION = 'Fine-tune top 20 layers, LR=1e-5'

# Stage 1: head only
model2, base2 = build_mobilenetv2_model(freeze_base=True, alpha=1.0)
model2.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss='binary_crossentropy', metrics=['accuracy'])
print('--- Stage 1: head only (10 epochs) ---')
history2a = model2.fit(train_ds, validation_data=val_ds, epochs=10,
                       callbacks=get_callbacks('mobilenetv2', f'{EXP_NUM}a'), verbose=1)

# Stage 2: unfreeze last 20 base layers
base2.trainable = True
for layer in base2.layers[:-20]:
    layer.trainable = False
print(f'Unfrozen layers: {sum(1 for l in base2.layers if l.trainable)}')
model2.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
               loss='binary_crossentropy', metrics=['accuracy'])
print('\n--- Stage 2: fine-tune top 20 layers (10 epochs, LR=1e-5) ---')
history2b = model2.fit(train_ds, validation_data=val_ds, epochs=10,
                       callbacks=get_callbacks('mobilenetv2', EXP_NUM), verbose=1)

metrics2 = evaluate_model(model2, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics2["accuracy"]}')
print(f'Precision: {metrics2["precision"]}')
print(f'Recall:    {metrics2["recall"]}')
print(f'F1-Score:  {metrics2["f1"]}')
print(f'AUC:       {metrics2["auc"]}')

combined2 = type('H', (), {'history': {
    k: history2a.history[k] + history2b.history[k] for k in history2a.history
}})()
plot_learning_curves(combined2, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P5_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics2['accuracy'], 'precision': metrics2['precision'],
    'recall': metrics2['recall'], 'f1': metrics2['f1'], 'auc': metrics2['auc'],
    'epochs': len(history2a.history['loss']) + len(history2b.history['loss']), 'notes': '',
})

**Interpretation**: *(Did fine-tuning 20 layers improve F1 over frozen base? Any instability in Stage 2?)*

---
## Experiment 3: Reduced Width Multiplier (alpha=0.75)

**Hypothesis**: Reducing alpha to 0.75 decreases all filter counts by 25%, producing a smaller and faster model. If the performance drop is minimal, this configuration would be preferable for deployment in resource-constrained clinical settings.

**Change made**: `alpha=0.75`, frozen base, LR=1e-3

In [ ]:
EXP_NUM         = 3
EXP_DESCRIPTION = 'Reduced width multiplier alpha=0.75, frozen base'
EPOCHS          = 10

model3, base3 = build_mobilenetv2_model(freeze_base=True, alpha=0.75)
model3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history3 = model3.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=get_callbacks('mobilenetv2', EXP_NUM), verbose=1,
)

metrics3 = evaluate_model(model3, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics3["accuracy"]}')
print(f'Precision: {metrics3["precision"]}')
print(f'Recall:    {metrics3["recall"]}')
print(f'F1-Score:  {metrics3["f1"]}')
print(f'AUC:       {metrics3["auc"]}')

plot_learning_curves(history3, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P5_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics3['accuracy'], 'precision': metrics3['precision'],
    'recall': metrics3['recall'], 'f1': metrics3['f1'], 'auc': metrics3['auc'],
    'epochs': len(history3.history['loss']), 'notes': '',
})

**Interpretation**: *(How much did alpha=0.75 degrade performance vs alpha=1.0? Is the accuracy–efficiency trade-off acceptable for clinical deployment?)*

---
## Experiment 4: Even Smaller Model (alpha=0.5)

**Hypothesis**: Halving the filter counts (alpha=0.5) creates a much lighter model. This explores the lower bound of MobileNetV2's accuracy–efficiency trade-off on this dataset and establishes whether the compact features are still sufficient for reliable malaria detection.

**Change made**: `alpha=0.5`, frozen base, LR=1e-3

In [ ]:
EXP_NUM         = 4
EXP_DESCRIPTION = 'Smallest width multiplier alpha=0.5, frozen base'
EPOCHS          = 10

model4, base4 = build_mobilenetv2_model(freeze_base=True, alpha=0.5)
model4.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history4 = model4.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=get_callbacks('mobilenetv2', EXP_NUM), verbose=1,
)

metrics4 = evaluate_model(model4, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics4["accuracy"]}')
print(f'Precision: {metrics4["precision"]}')
print(f'Recall:    {metrics4["recall"]}')
print(f'F1-Score:  {metrics4["f1"]}')
print(f'AUC:       {metrics4["auc"]}')

plot_learning_curves(history4, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P5_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics4['accuracy'], 'precision': metrics4['precision'],
    'recall': metrics4['recall'], 'f1': metrics4['f1'], 'auc': metrics4['auc'],
    'epochs': len(history4.history['loss']), 'notes': '',
})

**Interpretation**: *(How much accuracy is lost at alpha=0.5 vs 0.75 and 1.0? Is a lighter model still clinically viable?)*

---
## Experiment 5: Add Data Augmentation (alpha=1.0)

**Hypothesis**: Data augmentation should further regularise the head training, reducing the train–val gap and improving test generalisation. MobileNetV2's lightweight nature may make it more sensitive to augmentation than heavier models.

**Change made**: `use_augmentation=True`, `alpha=1.0`, frozen base

In [ ]:
EXP_NUM         = 5
EXP_DESCRIPTION = 'Data augmentation + frozen base, alpha=1.0'
EPOCHS          = 10

model5, base5 = build_mobilenetv2_model(freeze_base=True, alpha=1.0, use_augmentation=True)
model5.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history5 = model5.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=get_callbacks('mobilenetv2', EXP_NUM), verbose=1,
)

metrics5 = evaluate_model(model5, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics5["accuracy"]}')
print(f'Precision: {metrics5["precision"]}')
print(f'Recall:    {metrics5["recall"]}')
print(f'F1-Score:  {metrics5["f1"]}')
print(f'AUC:       {metrics5["auc"]}')

plot_learning_curves(history5, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P5_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics5['accuracy'], 'precision': metrics5['precision'],
    'recall': metrics5['recall'], 'f1': metrics5['f1'], 'auc': metrics5['auc'],
    'epochs': len(history5.history['loss']), 'notes': '',
})

**Interpretation**: *(Did augmentation improve train–val gap and F1 compared to Exp 1?)*

---
## Experiment 6: Longer Training with ReduceLROnPlateau (30 epochs)

**Hypothesis**: Allowing more epochs with an aggressive ReduceLROnPlateau schedule (patience=3, factor=0.3) gives MobileNetV2 more time to extract domain-specific features. The fine-grained LR reductions should find a better minimum than the 10-epoch configurations.

**Change made**: 30 epochs, augmentation on, `ReduceLROnPlateau(patience=3, factor=0.3)`

In [ ]:
EXP_NUM         = 6
EXP_DESCRIPTION = 'Longer training 30 epochs, augmentation, aggressive LR schedule'
EPOCHS          = 30

model6, base6 = build_mobilenetv2_model(freeze_base=True, alpha=1.0, use_augmentation=True)
model6.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

callbacks6 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=3, min_lr=1e-8, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='checkpoints/mobilenetv2_exp6.h5',
        monitor='val_accuracy', save_best_only=True, verbose=0),
]

history6 = model6.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=callbacks6, verbose=1,
)

metrics6 = evaluate_model(model6, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Stopped at epoch: {len(history6.history["loss"])}')
print(f'Accuracy:  {metrics6["accuracy"]}')
print(f'Precision: {metrics6["precision"]}')
print(f'Recall:    {metrics6["recall"]}')
print(f'F1-Score:  {metrics6["f1"]}')
print(f'AUC:       {metrics6["auc"]}')

plot_learning_curves(history6, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P5_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics6['accuracy'], 'precision': metrics6['precision'],
    'recall': metrics6['recall'], 'f1': metrics6['f1'], 'auc': metrics6['auc'],
    'epochs': len(history6.history['loss']), 'notes': '',
})

**Interpretation**: *(Did longer training with aggressive LR schedule improve over the 10-epoch runs? At which epoch did the model peak?)*

---
## Experiment 7: EfficientNetB0 as Alternative Base

**Hypothesis**: EfficientNetB0 uses compound scaling (depth, width, and resolution scaled together) and achieves higher accuracy than MobileNetV2 with a similar parameter count. Swapping the base provides a direct comparison of two efficiency-focused architectures on the same malaria dataset.

**Change made**: Base model swapped to `EfficientNetB0` with frozen weights; EfficientNetB0 handles its own preprocessing internally

In [ ]:
EXP_NUM         = 7
EXP_DESCRIPTION = 'EfficientNetB0 as alternative base (frozen)'
EPOCHS          = 10

# EfficientNetB0 handles preprocessing internally — pass raw [0,255] images
eff_base = tf.keras.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3),
)
eff_base.trainable = False

inputs  = tf.keras.Input(shape=(224, 224, 3))
x       = eff_base(inputs, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(128, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model7  = tf.keras.Model(inputs, outputs, name='efficientnetb0_transfer')

model7.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
model7.summary()

history7 = model7.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=get_callbacks('efficientnetb0', EXP_NUM), verbose=1,
)

metrics7 = evaluate_model(model7, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics7["accuracy"]}')
print(f'Precision: {metrics7["precision"]}')
print(f'Recall:    {metrics7["recall"]}')
print(f'F1-Score:  {metrics7["f1"]}')
print(f'AUC:       {metrics7["auc"]}')

plot_learning_curves(history7, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P5_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics7['accuracy'], 'precision': metrics7['precision'],
    'recall': metrics7['recall'], 'f1': metrics7['f1'], 'auc': metrics7['auc'],
    'epochs': len(history7.history['loss']), 'notes': '',
})

**Interpretation**: *(Did EfficientNetB0 outperform MobileNetV2 (Exp 1) with the same frozen-head setup? Which is the better lightweight base for malaria classification?)*

---
## 7. Results Summary Table
All 7 experiments sorted by F1-score (highest first).

In [ ]:
import pandas as pd
results_df = build_results_table(results_log)
pd.set_option('display.float_format', '{:.4f}'.format)
display(results_df)

## 8. Best Model — Detailed Evaluation
Identify the experiment with the highest F1-score and run the confusion matrix, ROC curve, and error analysis.

In [ ]:
exp_map = {
    1: (model1, metrics1),
    2: (model2, metrics2),
    3: (model3, metrics3),
    4: (model4, metrics4),
    5: (model5, metrics5),
    6: (model6, metrics6),
    7: (model7, metrics7),
}

best_exp_num = results_df.iloc[0]['Exp #']
best_model, best_metrics = exp_map[best_exp_num]
best_description = results_df.iloc[0]['Description']

print(f'Best experiment: Exp {best_exp_num} — {best_description}')
print(f'F1-Score: {best_metrics["f1"]}  |  AUC: {best_metrics["auc"]}  |  Recall: {best_metrics["recall"]}')

### Confusion Matrix
Plots the confusion matrix for the best MobileNetV2 model with Sensitivity and Specificity annotated to evaluate clinical diagnostic performance.

In [ ]:
plot_confusion_matrix(
    best_metrics, CLASS_NAMES,
    f'Best Model (Exp {best_exp_num}): {best_description}',
    save_path='figures/P5_best_confusion_matrix.png',
)

### ROC Curve
Plots the ROC curve and AUC for the best model — key for comparing MobileNetV2's discriminative ability against the heavier pretrained models and custom CNNs.

In [ ]:
plot_roc_curve(
    best_metrics,
    f'Best Model (Exp {best_exp_num}): {best_description}',
    save_path='figures/P5_best_roc_curve.png',
)

### Error Analysis
Displays misclassified test images to identify cell morphologies that MobileNetV2's lightweight depthwise separable features struggle to distinguish.

In [ ]:
error_analysis(best_model, test_ds, CLASS_NAMES, n_samples=12)

## 9. Model Summary & Report Notes

### Best configuration
- **Experiment**: *(fill in after running)*
- **Architecture**: MobileNetV2 alpha=X (ImageNet) + GAP + Dense(128) + Dropout(0.3) + sigmoid
- **Training strategy**: *(frozen only / two-stage fine-tune / longer training)*
- **Key hyperparameters**: *(LR, alpha, augmentation, epochs)*
- **Test metrics**: Accuracy=X, Precision=X, Recall=X, F1=X, AUC=X

### Clinical relevance
*(Discuss Recall/Sensitivity — MobileNetV2 is smaller and faster to deploy than VGG16/ResNet50. Is the accuracy trade-off acceptable for field deployment in resource-constrained settings? What is the clinical cost of any False Negatives?)*

### Observed patterns
- **Frozen vs fine-tuned**: *(Did fine-tuning top 20 layers consistently improve over frozen?)*
- **Alpha comparison**: *(How did 1.0 vs 0.75 vs 0.5 trade off accuracy for model size?)*
- **MobileNetV2 vs EfficientNetB0**: *(Which lightweight base performed better on this dataset?)*
- **Most impactful change**: *(Which experiment produced the largest F1 gain?)*

### Conclusion
*(Rank this model 1st–5th once all group members have run their experiments. Discuss whether MobileNetV2's efficiency advantage justifies any performance gap relative to VGG16/ResNet50.)*